# Mission 4.2 — PyTorch Non-Linear Ablation Lab (Module 04 PDF)
Run both experiments in Google Colab, keep all cell outputs, download `.ipynb` (File -> Download -> Download .ipynb), and submit to the Judge!


In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=600, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)
print("Tensors ready:", X_train_t.shape, X_test_t.shape)


In [ ]:
class SimpleNeuralNet(nn.Module):
    def __init__(self, hidden_dim=16, use_relu=True):
        super().__init__()
        if use_relu:
            self.network = nn.Sequential(
                nn.Linear(2, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, 2)
            )
        else:
            self.network = nn.Sequential(
                nn.Linear(2, hidden_dim),
                nn.Linear(hidden_dim, hidden_dim),
                nn.Linear(hidden_dim, 2)
            )
    def forward(self, x):
        return self.network(x)

def train_experiment(hidden_dim=16, use_relu=True, epochs=150):
    torch.manual_seed(42)
    model = SimpleNeuralNet(hidden_dim=hidden_dim, use_relu=use_relu)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        preds = torch.argmax(model(X_test_t), dim=1)
        acc = (preds == y_test_t).float().mean().item() * 100
    print(f"hidden={hidden_dim} | use_relu={use_relu} -> Loss={loss.item():.4f} | Accuracy={acc:.2f}%")

train_experiment(16, True)
train_experiment(16, False)  # Observe linear collapse!
train_experiment(64, True)   # Observe higher capacity!
